# NLP to SQL — baseline vs. targeted training data

Runs the full experiment: fine-tune **t5-small** on simple single-table queries (baseline), find where it fails, add training pairs that target those patterns, retrain, and compare both on the **same held-out test set** using **execution accuracy**.

**In Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (~5 min).

In [ ]:
!git clone -q https://github.com/UnnatiGupta0/Nlp-to-Sql.git
%cd Nlp-to-Sql
!pip install -q transformers sentencepiece accelerate pandas

## 1. The database
Five tables from `company_data_dirty.json`, cleaned (dropped `N/A` rows, normalised dates) and loaded into SQLite.

In [ ]:
from nlp2sql import load_db, make_pairs
conn, tables = load_db()
for name, df in tables.items():
    print(name, df.shape)
tables['employees'].head()

## 2. Training data
Question→SQL pairs are generated from templates.
- **simple**: single-table SELECT / WHERE / COUNT (what v1 trained on)
- **aggregation**: GROUP BY, COUNT(DISTINCT)
- **join**: 2- and 3-table JOINs
- **subquery**: NOT IN / IN with GROUP BY … HAVING

Pairs whose correct answer is empty are dropped, since an empty result is too easy to match by accident.

In [ ]:
pool = make_pairs(conn, tables, 1150, 1150, seed=42)
print(pool.category.value_counts())
pool.groupby('category').head(1)

## 3. Train baseline + improved models and evaluate
The test set (150 simple + 150 complex) is held out before any training, and the script asserts no test question appears in training.

In [ ]:
!python train.py

## 4. Results

In [ ]:
import json, pandas as pd
r = json.load(open('results/results.json'))
pd.DataFrame({k: {'execution_accuracy': v['execution_accuracy'], 'exact_match': v['exact_match'], **v['by_category']} for k, v in r.items() if k != 'test_set'})

## 5. Error analysis — what the baseline got wrong

In [ ]:
b = pd.read_csv('results/baseline_predictions.csv')
b[~b.exec_ok][['category', 'question', 'sql', 'pred']].groupby('category').head(2)

## 6. Try it

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
m = T5ForConditionalGeneration.from_pretrained('t5_sql_model'); t = T5Tokenizer.from_pretrained('t5_sql_model')
def ask(q):
    sql = t.decode(m.generate(**t('translate English to SQL: ' + q, return_tensors='pt'), max_new_tokens=128)[0], skip_special_tokens=True)
    print(sql)
    try: print(conn.execute(sql).fetchall()[:5])
    except Exception as e: print('failed:', e)
ask('How many female employees are from India?')
ask('Which male employees work on Python projects?')